# 🔥 PyTorch: Zero to Hero — A Guided Lab

Welcome to a complete, self-contained PyTorch course. You'll go from "what is a tensor?" to
training and deploying real neural networks — understanding every moving part.

**How this lab works**
- 📖 **Theory** (detailed) → 🧠 **Mental model** → 🖼️ **ASCII diagram** → 🔬 **Worked example**
  → ⚡ **Pro tips** → ⚠️ **Common traps** → ✏️ **Your Turn** → ✅ **Solution**.
- Run every cell. Attempt exercises before revealing solutions.

**Prerequisite:** the NumPy Zero-to-Master lab (tensors are NumPy arrays with superpowers).

**Install:** `pip install torch`

**Roadmap**
1. Tensors — creation, shapes, dtypes, devices
2. Tensor operations & broadcasting
3. Autograd — automatic differentiation
4. From manual gradient descent → `nn.Module`
5. The canonical training loop
6. Datasets & DataLoaders
7. A real regression project
8. A real classification project
9. Saving/loading & inference
10. 🏆 Capstone: end-to-end model


In [ ]:
import torch
import torch.nn as nn
import numpy as np
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

---
## Chapter 1 — Tensors

📖 **Theory.** A **tensor** is PyTorch's core data structure: an n-dimensional array, much
like a NumPy array, but with two superpowers:
1. It can live on a **GPU** for massive parallel speed.
2. It can **track gradients** for automatic differentiation (the basis of training).

Tensors have a **shape** (size per dimension), a **dtype** (float32, int64, bool…), and a
**device** (cpu or cuda).

🧠 **Mental model.** Think of tensor rank as nested boxes:
- rank-0 = a single number (scalar)
- rank-1 = a list of numbers (vector)
- rank-2 = a table (matrix)
- rank-3 = a stack of tables (e.g. a batch of images)

🖼️ **Diagram — tensor ranks**
```
scalar (0-D)   vector (1-D)      matrix (2-D)         3-D tensor
   5           [5, 2, 9]        [[1, 2, 3],        [ [[..],[..]],
                                 [4, 5, 6]]          [[..],[..]] ]
 shape ()      shape (3,)       shape (2, 3)        shape (2, 2, 3)
```


In [ ]:
scalar = torch.tensor(5.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"{name:7s} shape={tuple(t.shape)} ndim={t.ndim} dtype={t.dtype}")

# Common creation patterns
print("\nzeros:", torch.zeros(2,3).shape)
print("ones:", torch.ones(2,3).shape)
print("rand [0,1):", torch.rand(2,2))
print("randn (normal):", torch.randn(2,2))
print("arange:", torch.arange(0, 10, 2))
print("from numpy:", torch.from_numpy(np.array([1,2,3])))

⚡ **Pro tip.** Default float dtype is `float32` — the sweet spot of speed vs. precision for
deep learning. Only use `float64` when you specifically need it.

⚠️ **Common trap.** Mixing dtypes (e.g. a `float32` tensor with a `float64` one) throws an
error. And integer tensors can't hold gradients — parameters must be float.

### ✏️ Your Turn 1.1
1. Create a `(3, 4)` tensor of random normal values.
2. Print its shape, dtype, and number of elements (`.numel()`).
3. Create a tensor of the numbers 0–9 as **float32**.

In [ ]:
t = None
nums = None
# print shape, dtype, numel of t; create nums as float32 0-9


✅ **Solution**
```python
t = torch.randn(3, 4)
print(tuple(t.shape), t.dtype, t.numel())
nums = torch.arange(10, dtype=torch.float32)
```

---
## Chapter 2 — Tensor Operations & Broadcasting

📖 **Theory.** Tensor math is elementwise and vectorized (just like NumPy). Broadcasting lets
you combine different-but-compatible shapes. Two crucial distinctions:
- `*` is **elementwise** multiply; `@` (or `torch.matmul`) is **matrix** multiply.
- Reshaping: `.view()` / `.reshape()` change shape; `.T` / `.transpose()` swap dims;
  `.unsqueeze(d)` adds a size-1 dim; `.squeeze()` removes size-1 dims.

🖼️ **Diagram — matmul shapes**
```
   (2 x 3)        (3 x 4)          (2 x 4)
  [[. . .]      [[. . . .]        [[. . . .]
   [. . .]]  @   [. . . .]    =    [. . . .]]
                 [. . . .]]
   inner dims (3) must match; outer dims give the result shape
```


In [ ]:
a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])   # (2,3)
b = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])  # (3,2)

print("elementwise a*a:\n", a*a)
print("matmul a@b (2x2):\n", a @ b)

# broadcasting: add a per-column bias (shape (3,)) to every row
bias = torch.tensor([10., 20., 30.])
print("broadcast add:\n", a + bias)

# reshaping tools
x = torch.arange(6.)
print("\nview 2x3:", x.view(2,3).shape)
print("unsqueeze(0):", x.unsqueeze(0).shape)   # (1,6)
print("unsqueeze(1):", x.unsqueeze(1).shape)   # (6,1)

⚡ **Pro tip.** `.view()` needs contiguous memory and is free (no copy); `.reshape()` is
safer (copies if needed). When unsure, use `.reshape()`.

⚠️ **Common trap.** Shape mismatches are the #1 PyTorch bug. Before any matmul, say the
shapes out loud: "(2×3) @ (3×2) → inner 3s match → (2×2)." If inner dims differ, you get a
`RuntimeError`.

### ✏️ Your Turn 2.1
1. Make `A` shape `(4, 3)` and `B` shape `(3, 2)`; compute `A @ B` and confirm shape `(4,2)`.
2. Take a `(5,)` vector and turn it into a column vector `(5, 1)` two ways: `.unsqueeze` and `.reshape`.

In [ ]:
A = None; B = None; AB = None
col1 = None; col2 = None
print(AB.shape if AB is not None else None)

✅ **Solution**
```python
A = torch.randn(4,3); B = torch.randn(3,2)
AB = A @ B                      # (4,2)
v = torch.arange(5.)
col1 = v.unsqueeze(1)           # (5,1)
col2 = v.reshape(5,1)
```

---
## Chapter 3 — Autograd: Automatic Differentiation

📖 **Theory.** Training a network means adjusting parameters to reduce a **loss**. To know
*which way* to adjust each parameter, we need the **gradient** of the loss w.r.t. that
parameter. PyTorch computes these automatically via **autograd**.

When a tensor has `requires_grad=True`, PyTorch records every operation into a **computation
graph**. Calling `.backward()` on the final loss walks that graph backward (the chain rule),
filling each parameter's `.grad`.

🧠 **Mental model.** Autograd is like a flight recorder: it logs every operation on the way
*forward*, then replays them in reverse to compute how each input influenced the output.

🖼️ **Diagram — forward builds the graph, backward fills gradients**
```
 forward  ---------------------------->
   w ──►(×)──► wx ──►(+)──► y ──►(loss)      loss = (y - target)^2
   x ──►         b ──►
 <----------------------------  backward()
   dloss/dw ◄─ dloss/dy ◄─ dloss/dy ... (chain rule fills .grad on w, b)
```


In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)
target = torch.tensor(10.0)

# forward pass -- builds the graph
y = w * x + b            # 2*3 + 1 = 7
loss = (y - target)**2   # (7 - 10)^2 = 9

# backward pass -- computes gradients
loss.backward()
print("y =", y.item(), " loss =", loss.item())
print("dloss/dw =", w.grad.item())   # = 2*(y-target)*x = 2*(-3)*3 = -18
print("dloss/db =", b.grad.item())   # = 2*(y-target)*1 = -6

⚡ **Pro tip.** Wrap inference/eval code in `with torch.no_grad():` so PyTorch doesn't build
the graph — saving memory and time when you don't need gradients.

⚠️ **Common trap.** Gradients **accumulate** by default. If you call `.backward()` in a loop
without zeroing grads first, they pile up and corrupt training. You must `zero_grad()` each step.

### ✏️ Your Turn 3.1
Let `f = w**2 + 3*w + 1` with `w = torch.tensor(4.0, requires_grad=True)`.
1. Compute `f.backward()` and read `w.grad`.
2. The analytic derivative is `2w + 3`. Confirm they match at w=4 (should be 11).

In [ ]:
w = torch.tensor(4.0, requires_grad=True)
f = None
# f.backward(); print(w.grad)


✅ **Solution**
```python
w = torch.tensor(4.0, requires_grad=True)
f = w**2 + 3*w + 1
f.backward()
print(w.grad.item())   # 2*4 + 3 = 11.0
```

---
## Chapter 4 — From Manual Gradient Descent to `nn.Module`

📖 **Theory.** **Gradient descent** updates each parameter a small step *against* its
gradient: `param -= learning_rate * param.grad`. Repeat until the loss stops improving.

Doing this by hand teaches intuition, but real networks have thousands of parameters. PyTorch
gives us:
- **`nn.Module`** — a container that holds parameters and defines a `forward()` method.
- **`nn.Linear`, `nn.ReLU`, `nn.Sequential`** — prebuilt layers.
- **optimizers** (`torch.optim`) — do the parameter updates for you.

🖼️ **Diagram — a small network**
```
input        Linear(3→4)   ReLU      Linear(4→1)   output
 x (3) ───►  [ W1, b1 ] ──► max(0,·) ──► [ W2, b2 ] ──► ŷ (1)
```


In [ ]:
# --- Manual gradient descent: fit y = 2x + 1 from data ---
torch.manual_seed(0)
x = torch.linspace(-1, 1, 50).unsqueeze(1)   # (50,1)
y = 2*x + 1 + 0.1*torch.randn_like(x)        # noisy target

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1
for step in range(200):
    pred = x*w + b
    loss = ((pred - y)**2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_(); b.grad.zero_()
print(f"Learned w={w.item():.3f} (true 2), b={b.item():.3f} (true 1)")

In [ ]:
# --- The same thing with nn.Module + optimizer (how you'll really do it) ---
model = nn.Sequential(nn.Linear(1, 1))
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for step in range(200):
    optimizer.zero_grad()
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()

W = model[0].weight.item(); B = model[0].bias.item()
print(f"nn.Module learned w={W:.3f}, b={B:.3f}")

🧠 **Mental model.** The optimizer + `nn.Module` do exactly what the manual loop did — they
just hide the bookkeeping. The 5-step rhythm is always the same (next chapter).

### ✏️ Your Turn 4.1
Build a 2-layer network with `nn.Sequential`: `Linear(4→8) → ReLU → Linear(8→1)`. Pass a
random `(10, 4)` batch through it and confirm the output shape is `(10, 1)`.

In [ ]:
net = None
out = None
print(out.shape if out is not None else None)

✅ **Solution**
```python
net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,1))
out = net(torch.randn(10,4))   # (10,1)
```

---
## Chapter 5 — The Canonical Training Loop

📖 **Theory.** Almost every PyTorch training loop is these 5 steps, repeated per batch:

🖼️ **Diagram — the 5-step rhythm**
```
   ┌─────────────────────────────────────────────┐
   │ 1. optimizer.zero_grad()   # clear old grads │
   │ 2. preds = model(x)        # forward pass    │
   │ 3. loss = loss_fn(preds,y) # measure error   │
   │ 4. loss.backward()         # compute grads   │
   │ 5. optimizer.step()        # update weights  │
   └─────────────────────────────────────────────┘
        repeat for every batch, every epoch
```

⚠️ **Common trap.** Forgetting `zero_grad()` (grads accumulate) or forgetting `.step()`
(weights never update). If your loss is flat or exploding, check these first.


In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 3)
true_w = torch.tensor([[1.5],[-2.0],[0.5]])
y = X @ true_w + 0.1*torch.randn(200,1)

model = nn.Linear(3, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for epoch in range(100):
    optimizer.zero_grad()      # 1
    preds = model(X)           # 2
    loss = loss_fn(preds, y)   # 3
    loss.backward()            # 4
    optimizer.step()           # 5
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")
print("\nlearned weights:", model.weight.data.numpy().round(2), "(true 1.5,-2,0.5)")

⚡ **Pro tip — sanity check.** Before a long training run, confirm your model can *overfit*
a tiny batch (loss → ~0). If it can't, there's a bug — fix it before wasting time on full data.

### ✏️ Your Turn 5.1
Take the loop above and switch the optimizer from `Adam` to plain `SGD` with `lr=0.1`. Run it
and compare the final loss. (Both should learn; Adam usually converges faster.)

In [ ]:
# Copy the loop, change optimizer to torch.optim.SGD(model.parameters(), lr=0.1)


✅ **Solution**
```python
model = nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
# ...identical 5-step loop...
```

---
## Chapter 6 — Datasets & DataLoaders

📖 **Theory.** Real data is too big to feed all at once. A **Dataset** wraps your data (with
`__len__` and `__getitem__`); a **DataLoader** serves it in shuffled **mini-batches**.

🖼️ **Diagram — batching**
```
 full data (200 rows)
 └─ DataLoader(batch_size=32, shuffle=True)
     ├─ batch 1: rows [ shuffled 32 ]
     ├─ batch 2: rows [ shuffled 32 ]
     └─ ...  (one epoch = all batches once)
```

🧠 **Mental model.** Shuffling each epoch stops the model from memorizing data order; batching
gives smoother, memory-friendly updates.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(200, 3)
y = (X.sum(dim=1, keepdim=True) > 0).float()   # simple binary target

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("num batches per epoch:", len(loader))
for xb, yb in loader:
    print("first batch shapes:", xb.shape, yb.shape)
    break

⚠️ **Common trap.** Always shuffle the **training** loader (`shuffle=True`) but **not** the
validation/test loader (you want stable, repeatable evaluation).

### ✏️ Your Turn 6.1
Create a `TensorDataset` from `X2 = torch.randn(120, 5)` and `y2 = torch.randn(120, 1)`, wrap
it in a `DataLoader` with `batch_size=16, shuffle=True`, and print how many batches it yields.

In [ ]:
X2 = torch.randn(120,5); y2 = torch.randn(120,1)
loader2 = None
print(len(loader2) if loader2 is not None else None)

✅ **Solution**
```python
loader2 = DataLoader(TensorDataset(X2, y2), batch_size=16, shuffle=True)
print(len(loader2))   # 120/16 -> 8 batches (last one smaller)
```

---
## Chapter 7 — Real Project: Regression (predict house prices)

📖 **Theory.** We'll predict a continuous value. Key real-world steps that beginners skip:
- **Normalize** inputs (and often the target) — unscaled features make training unstable.
- **Split** train/test — evaluate on unseen data.
- **Un-normalize** predictions to get answers in real units.


In [ ]:
torch.manual_seed(0); np.random.seed(0)
n = 1000
size = np.random.uniform(500, 3500, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 50, n)
price = size*150 + bedrooms*8000 - age*300 + np.random.normal(0, 15000, n)

X = np.stack([size, bedrooms, age], axis=1).astype(np.float32)
y = price.astype(np.float32).reshape(-1,1)

# normalize
Xm, Xs = X.mean(0), X.std(0)
ym, ys = y.mean(), y.std()
Xn = torch.tensor((X-Xm)/Xs); yn = torch.tensor((y-ym)/ys)

Xtr, Xte = Xn[:800], Xn[800:]
ytr, yte = yn[:800], yn[800:]
print("train/test:", Xtr.shape, Xte.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)

model = nn.Sequential(nn.Linear(3,16), nn.ReLU(), nn.Linear(16,1))
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(30):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = loss_fn(model(Xte), yte).item()
        print(f"epoch {epoch:2d}  test_loss {test_loss:.4f}")

### ✏️ Your Turn 7.1
Predict the price of a **2000 sqft, 3-bedroom, 10-year-old** house. Remember: normalize the
input with `Xm/Xs`, run the model in eval mode, then un-normalize with `ym/ys`.

In [ ]:
new_house = np.array([[2000, 3, 10]], dtype=np.float32)
predicted_price = None
print(predicted_price)

✅ **Solution**
```python
xn = torch.tensor((new_house - Xm)/Xs)
model.eval()
with torch.no_grad():
    pred_norm = model(xn).item()
predicted_price = pred_norm*ys + ym
print(f"${predicted_price:,.0f}")
```

---
## Chapter 8 — Real Project: Classification

📖 **Theory.** For classification the output layer produces one **logit per class**, and we
use **CrossEntropyLoss** (which applies softmax internally — so **don't** add softmax to your
model). Labels are integer class indices.

🖼️ **Diagram — classification head**
```
 features ─► Linear(→3) ─► logits [2.1, -0.5, 0.8]
                                   │ (CrossEntropyLoss applies softmax)
                                   ▼
                        probs [0.63, 0.05, 0.32] ─► argmax = class 0
```


In [ ]:
from sklearn.datasets import make_classification
Xc, yc = make_classification(n_samples=600, n_features=8, n_classes=3, n_informative=5, random_state=0)
Xc = torch.tensor(Xc, dtype=torch.float32)
yc = torch.tensor(yc, dtype=torch.long)          # integer class labels
Xtr, Xte, ytr, yte = Xc[:480], Xc[480:], yc[:480], yc[480:]

clf = nn.Sequential(nn.Linear(8,32), nn.ReLU(), nn.Linear(32,3))  # 3 logits, NO softmax
opt = torch.optim.Adam(clf.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(60):
    clf.train(); opt.zero_grad()
    loss = loss_fn(clf(Xtr), ytr)
    loss.backward(); opt.step()

clf.eval()
with torch.no_grad():
    preds = clf(Xte).argmax(dim=1)
    acc = (preds == yte).float().mean().item()
print(f"test accuracy: {acc:.3f}")

⚠️ **Common trap.** Adding a `softmax` before `CrossEntropyLoss` double-applies it and hurts
training. `CrossEntropyLoss` expects **raw logits**. Also: labels must be `long` (int64), not
one-hot, and not float.

### ✏️ Your Turn 8.1
After training, compute the model's **predicted class** for the first 5 test samples and
compare to the true labels `yte[:5]`.

In [ ]:
# predict first 5, compare to yte[:5]


✅ **Solution**
```python
clf.eval()
with torch.no_grad():
    p5 = clf(Xte[:5]).argmax(dim=1)
print("pred:", p5.tolist(), "true:", yte[:5].tolist())
```

---
## Chapter 9 — Saving, Loading & Inference

📖 **Theory.** Save the **state_dict** (the learned parameters), not the whole model object —
it's the portable, recommended way.
- Save: `torch.save(model.state_dict(), "model.pt")`
- Load: rebuild the same architecture, then `model.load_state_dict(torch.load("model.pt"))`
- Always call `model.eval()` before inference (disables dropout/batchnorm training behavior).


In [ ]:
# save
torch.save(clf.state_dict(), "/tmp/classifier.pt")

# load into a fresh model with the SAME architecture
clf2 = nn.Sequential(nn.Linear(8,32), nn.ReLU(), nn.Linear(32,3))
clf2.load_state_dict(torch.load("/tmp/classifier.pt"))
clf2.eval()

with torch.no_grad():
    same = (clf2(Xte).argmax(1) == clf(Xte).argmax(1)).all().item()
print("reloaded model matches original:", same)

⚠️ **Common trap.** Loading a state_dict requires the **exact same architecture**. If you
change layer sizes, loading fails. Keep your model definition alongside the weights.

### ✏️ Your Turn 9.1
Save the `model` (the regression net from Ch.7) to `/tmp/reg.pt`, rebuild it, load the
weights, and confirm it produces the same prediction on `Xte[:1]`.

In [ ]:
# save regression model, reload, compare predictions on Xte[:1]


✅ **Solution**
```python
torch.save(model.state_dict(), "/tmp/reg.pt")
m2 = nn.Sequential(nn.Linear(3,16), nn.ReLU(), nn.Linear(16,1))
m2.load_state_dict(torch.load("/tmp/reg.pt")); m2.eval()
with torch.no_grad():
    print(torch.allclose(m2(Xte[:1]), model(Xte[:1])))
```

---
## 🏆 Chapter 10 — Capstone: End-to-End Model

Build, train, evaluate, and save a classifier **from scratch**, using everything you learned.

**Task:** classify the classic Iris-style problem below (4 features, 3 classes).
Do the full pipeline yourself before revealing the solution.

In [ ]:
from sklearn.datasets import load_iris
data = load_iris()
Xi = torch.tensor(data.data, dtype=torch.float32)
yi = torch.tensor(data.target, dtype=torch.long)
# shuffle + split
torch.manual_seed(0)
perm = torch.randperm(len(Xi))
Xi, yi = Xi[perm], yi[perm]
split = 120
Xtr, Xte = Xi[:split], Xi[split:]
ytr, yte = yi[:split], yi[split:]
print("train/test:", Xtr.shape, Xte.shape, "classes:", yi.unique().tolist())

### ✏️ Capstone Tasks
1. **Normalize** features using the training mean/std (apply the *same* stats to test).
2. Build a network: `Linear(4→16) → ReLU → Linear(16→3)`.
3. Train with `CrossEntropyLoss` + Adam for ~100 epochs.
4. Report **test accuracy** (aim for > 0.90).
5. **Save** the model's state_dict to `/tmp/iris.pt`.

In [ ]:
# Your full pipeline here


✅ **Capstone Solution**
```python
# 1. normalize with TRAIN stats
mu, sd = Xtr.mean(0), Xtr.std(0)
Xtr_n = (Xtr - mu)/sd
Xte_n = (Xte - mu)/sd

# 2. model
model = nn.Sequential(nn.Linear(4,16), nn.ReLU(), nn.Linear(16,3))
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

# 3. train
for epoch in range(100):
    model.train(); opt.zero_grad()
    loss = loss_fn(model(Xtr_n), ytr)
    loss.backward(); opt.step()

# 4. evaluate
model.eval()
with torch.no_grad():
    acc = (model(Xte_n).argmax(1) == yte).float().mean().item()
print(f"test accuracy: {acc:.3f}")

# 5. save
torch.save(model.state_dict(), "/tmp/iris.pt")
```

🎉 **You're a PyTorch practitioner now!** You understand tensors, autograd, the training loop,
data loading, real regression & classification, and model persistence. Everything else in deep
learning (CNNs, RNNs, Transformers) is built on exactly these foundations.

---
### 📌 Function Quick-Reference
**Tensors:** `tensor, zeros, ones, rand, randn, arange, from_numpy, .shape, .dtype, .numel, .to(device)`
**Ops:** `+ - * / @, matmul, .view, .reshape, .T, .transpose, .unsqueeze, .squeeze, .mean, .sum`
**Autograd:** `requires_grad, .backward, .grad, torch.no_grad()`
**Layers:** `nn.Linear, nn.ReLU, nn.Sequential, nn.Module`
**Loss/optim:** `nn.MSELoss, nn.CrossEntropyLoss, optim.SGD, optim.Adam, zero_grad, step`
**Data:** `TensorDataset, DataLoader(batch_size, shuffle)`
**Train/eval:** `model.train(), model.eval(), .argmax`
**Persistence:** `torch.save, torch.load, state_dict, load_state_dict`
